താഴെ കൊടുത്തിരിക്കുന്ന നോട്ട്‌ബുക്ക്‌കൾ പ്രവർത്തിപ്പിക്കാൻ, നിങ്ങൾ ഇതുവരെ ചെയ്യാതിരുന്നുവെങ്കിൽ, `text-embedding-ada-002` എന്ന ബേസ് മോഡൽ ഉപയോഗിക്കുന്ന ഒരു മോഡൽ വിന്യസിച്ച്, .env ഫയലിൽ വിന്യാസത്തിന്റെ പേര് `AZURE_OPENAI_EMBEDDINGS_ENDPOINT` ആയി സജ്ജീകരിക്കണം  


In [ ]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2024-10-21",
  azure_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"


അടുത്തതായി, നാം Embedding Index ഒരു Pandas Dataframe ൽ ലോഡ് ചെയ്യാൻ പോകുന്നു. Embedding Index `embedding_index_3m.json` എന്ന JSON ഫയലിൽ സ്റ്റോർ ചെയ്തിട്ടുണ്ട്. Embedding Index ഒക്ടോബർ 2023 അവസാനം വരെ ഉള്ള YouTube ട്രാൻസ്ക്രിപ്റ്റുകൾക്ക് വേണ്ടി Embeddings അടങ്ങിയിരിക്കുന്നു.


In [ ]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

പിന്നീട്, നാം `get_videos` എന്നോർ ഫങ്ഷൻ സൃഷ്ടിക്കാൻ പോകുകയാണ്, അത് എൻബെഡിങ്ങ് ഇൻഡെക്സിൽ ക്വറിയുമായി സാദൃശ്യമുള്ളത് തിരയുന്നതാണ്. ഫങ്ഷൻ ക്വറിയിനോട് ഏറ്റവും സാമ്യമുള്ള മുകളിൽ 5 വീഡിയോസ് തിരിച്ച് നൽകും. ഫങ്ഷൻ പ്രവർത്തിക്കുന്ന വിധം ഇതാണ procéഡ്:

1. ആദ്യമായി, എൻബെഡിങ്ങ് ഇൻഡെക്സിന്റെ ഒരു കോപ്പി സൃഷ്ടിക്കുന്നു.
2. തുടര്‍ന്ന, OpenAI എൻബെഡിങ്ങ് API ഉപയോഗിച്ച് ക്വറിയുടെ എൻബെഡിങ്ങ് കണക്കാക്കുന്നു.
3. പിന്നിട, എൻബെഡിങ്ങ് ഇൻഡെക്സിൽ `similarity` എന്ന പുതിയ ഒരു കോളം സൃഷ്ടിക്കുന്നു. `similarity` കോളത്തിലുണ്ട് ക്വറി എൻബെഡിങ്ങും ഓരോ വീഡിയോ സെഗ്മന്റിന്റെയും എൻബെഡിങ്ങും തമ്മിലുള്ള കോസൈൻ സാദൃശ്യത്തിന്റെ മൂല്യം.
4. തുടർന്ന്, `similarity` കോളം അടിസ്ഥാനമാക്കി എൻബെഡിങ്ങ് ഇൻഡെക്സ് ഫിൽട്ടർ ചെയ്യുന്നു. കോസൈൻ സാദൃശ്യ മൂല്യം 0.75യ്‌ക്കോ അതിനുമേൽ ഉള്ള വീഡിയോകൾ മാത്രമായി ഉൾപ്പെടുത്തുന്നു.
5. ഒടുവിൽ, `similarity` കോളം പ്രകാരം എൻബെഡിങ്ങ് ഇൻഡെക്സ് സോർട്ട് ചെയ്ത് മുകളിൽ 5 വീഡിയോകൾ തിരിച്ച് നൽകുന്നു.


In [ ]:
def cosine_similarity(a, b):
    if len(a) > len(b):
        b = np.pad(b, (0, len(a) - len(b)), 'constant')
    elif len(b) > len(a):
        a = np.pad(a, (0, len(b) - len(a)), 'constant')
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

ഈ ഫംഗ്ഷൻ വളരെ ലളിതമാണ്, ഇത് സേർച്ചു ക്വറിയുടെ ഫലങ്ങൾ മാത്രം പ്രിന്റ് ചെയ്യുന്നു.


In [ ]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. ആദ്യം, എംബെഡിങ് ഇൻഡക്സ് പാണ്ടാസ് ഡാറ്റാഫ്രെയിമിലേക്ക് ലോഡുചെയ്യുന്നു.
2. അതിന് ശേഷം, ഉപയോക്താവിന് ഒരു ചോദ്യം നൽകാൻ പ്രാപ്തമാക്കുന്നു.
3. പിന്നെയാണ് `get_videos` ഫംഗ്ഷൻ എംബെഡിങ് ഇൻഡക്സിൽ ചോദ്യം തിരയാൻ വിളിക്കുന്നത്.
4. അവസാനം, ഫലങ്ങൾ ഉപയോക്താവിന് കാണിക്കാൻ `display_results` ഫംഗ്ഷൻ വിളിക്കും.
5. തുടർന്ന് ഉപയോക്താവിന് മറ്റൊരു ചോദ്യം നൽകാൻ പ്രാപ്തമാക്കുന്നു. ഉപയോക്താവ് `exit` നൽകുന്നതുവരെ ഈ പ്രക്രിയ തുടരുന്നു.

![](../../../../translated_images/ml/notebook-search.1e320b9c7fcbb0bc.webp)

നിങ്ങളെ ഒരു ചോദ്യം നൽകാൻ പ്രാപ്തമാക്കും. ഒരു ചോദ്യം നൽകുക, എന്റർ അമർത്തുക. അപേക്ഷ കൊടുത്തുവന്ന ചോദ്യം സംബന്ധിച്ച വീഡിയോകളുടെ പട്ടിക പ്രസ്തുതിക്കും. ചോദ്യം ഉത്തരം കിട്ടുന്ന വീഡിയോയിലെ ഇടം കാണിക്കുന്ന ഒരു ലിങ്കും അപേക്ഷ നൽകും.

പരീക്ഷിക്കാൻ ചില ചോദ്യങ്ങൾ ഇതാ:

- Azure Machine Learning എന്നത് എന്താണ്?
- കോൺവല്യൂഷണൽ ന്യൂറൽ നെറ്റ്‌വർക്കുകൾ എങ്ങനെ പ്രവർത്തിക്കുന്നു?
- ഒരു ന്യൂറൽ നെറ്റ്‌വർക് എന്താണ്?
- Azure Machine Learning-നൊപ്പം Jupyter Notebooks ഉപയോഗിക്കാമോ?
- ONNX എന്നത് എന്താണ്?


In [ ]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
